# Урок 11 - Протокол контекста модели (MCP)

The **Протокол контекста модели (MCP)** — это открытый стандарт, который позволяет агентам динамически обнаруживать и использовать инструменты, ресурсы и источники данных во время выполнения. Вместо того чтобы жёстко встраивать инструменты в агента, MCP позволяет агентам подключаться к внешним серверам, которые по требованию предоставляют возможности.

В этом уроке вы узнаете:
- Что такое MCP и почему это важно для агентных систем
- Как работает клиент-серверная архитектура MCP
- Как создавать агентов, которые используют обнаружение инструментов в стиле MCP


## Настройка

**Требования:**
- Проект Azure AI Foundry с развернутой моделью
- Выполните `az login` для аутентификации


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Что такое Model Context Protocol (MCP)?

MCP определяет стандартный способ, с помощью которого агенты ИИ могут обнаруживать и взаимодействовать с внешними инструментами и источниками данных:

- **MCP Server**: Предоставляет инструменты, ресурсы и подсказки через стандартный протокол
- **MCP Client**: Среда выполнения агента, которая подключается к серверам и обнаруживает доступные возможности
- **Dynamic Discovery**: Агентам не нужны жёстко заданные инструменты — они обнаруживают доступные возможности во время выполнения

Это мощный подход для построения расширяемых систем агентов, где новые возможности можно добавлять без изменения кода агента.


## Как работает MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Агент (MCP client) устанавливает соединение с MCP server
2. Сервер отвечает списком доступных инструментов и их схем
3. Затем агент может вызывать любой обнаруженный инструмент в ходе своего рассуждения
4. Результаты возвращаются через тот же протокол


## Симуляция обнаружения инструментов MCP

Поскольку реальный сервер MCP требует запущенного серверного процесса, мы продемонстрируем этот шаблон, используя функции `@tool`, которые симулируют то, что предоставлял бы подключённый к MCP сервис размещения.

В рабочей среде эти инструменты обнаруживались бы динамически с сервера MCP, а не определялись локально.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Создание агента с инструментами в стиле MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP в производственной среде

В производственной среде MCP обеспечивает мощные шаблоны:

- **Динамическое обнаружение инструментов**: Агенты подключаются к MCP-серверам и обнаруживают инструменты во время выполнения
- **Разделённая архитектура**: Поставщики инструментов могут обновляться независимо от агента
- **Совместное использование между организациями**: Команды могут предоставлять возможности через MCP-серверы, которые может использовать любой агент
- **Поддержка Microsoft Agent Framework**: MAF имеет встроенную поддержку клиента MCP через интеграцию `mcp`

Чтобы использовать реальный MCP-сервер с MAF, вы подключитесь через `hosted_mcp_tool()` или интеграцию клиента MCP.

**Узнать больше:**
- [Спецификация MCP](https://modelcontextprotocol.io/)
- [Поддержка MCP в Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Резюме

В этом уроке вы узнали:
- **MCP** — открытый стандарт для динамического обнаружения инструментов между агентами и поставщиками инструментов
- **Клиент-серверная архитектура** позволяет агентам обнаруживать возможности во время выполнения
- MCP обеспечивает **расширяемые, слабо связанные системы агентов**, в которых инструменты можно добавлять без изменения кода
- Microsoft Agent Framework предоставляет **встроенную поддержку MCP** для использования в производственной среде


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Отказ от ответственности:
Этот документ был переведён с помощью сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Хотя мы прилагаем усилия для обеспечения точности, имейте в виду, что автоматические переводы могут содержать ошибки или неточности. Оригинальный документ на языке оригинала следует считать авторитетным источником. Для получения критически важной информации рекомендуется профессиональный перевод, выполненный человеком. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
